# 01. 컨텍스트와 토큰 예산

모델 호출에는 입력과 출력이 함께 들어갈 공간이 필요합니다. 이 노트북에서는 입력 길이를 호출 전에 추정하고, 실제 API 사용량과 비교합니다.

## 실행 환경 준비

첫 코드 셀은 노트북에서 사용할 패키지 버전을 맞춥니다. 설치가 끝나면 이어지는 셀을 순서대로 실행합니다.

In [1]:
%pip install -q "openai==3.8.0" "python-dotenv==1.2.3" "tiktoken==0.14.0"

Note: you may need to restart the kernel to use updated packages.


## 1. API 연결 준비

아래 셀은 현재 작업 폴더에서 상위 폴더로 올라가며 `.env`를 찾습니다. 키 값은 화면에 출력하지 않습니다.

In [2]:
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
import os

env_path = find_dotenv(usecwd=True)
if not env_path:
    raise FileNotFoundError("현재 폴더의 상위 경로에서 .env를 찾지 못했습니다.")
load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env에 OPENAI_API_KEY가 없습니다.")
client = OpenAI()
MODEL = "gpt-5.4-mini"
print("API 실습 준비 완료")

API 실습 준비 완료


## 2. 출력 공간을 먼저 예약합니다

전체 예산에서 출력 예약분과 안전 여유를 빼면 입력에 사용할 수 있는 범위가 나옵니다.

In [3]:
def input_budget(total: int, output_reserve: int, safety_margin: int) -> int:
    available = total - output_reserve - safety_margin
    if available < 0:
        raise ValueError("출력 예약분과 여유가 전체 예산보다 큽니다.")
    return available

available = input_budget(4096, 300, 128)
print("입력에 사용할 수 있는 토큰:", available)

입력에 사용할 수 있는 토큰: 3668


## 3. 로컬 추정값을 확인합니다

토크나이저의 추정값은 호출 전 차단에 쓰고, API의 `usage`는 호출 후 기록에 씁니다.

In [4]:
import tiktoken

question = "미르시립도서관의 정기 휴관일은 언제인가요?"
evidence = "[E01 p.2] 정기 휴관일은 매주 월요일입니다."
prompt = f"근거:\n{evidence}\n\n질문: {question}"
encoding = tiktoken.get_encoding("o200k_base")
estimated = len(encoding.encode(prompt))
print("로컬 추정 입력 토큰:", estimated)
print("예산 안에 있음:", estimated <= available)

로컬 추정 입력 토큰: 42
예산 안에 있음: True


## 4. 실제 API 사용량을 비교합니다

답의 내용과 `usage`를 함께 확인합니다. 출력은 실행마다 달라질 수 있습니다.

In [5]:
response = client.responses.create(
    model=MODEL,
    instructions="근거에 있는 정보만 사용하고 출처를 표시하세요.",
    input=prompt,
    max_output_tokens=300,
    reasoning={"effort": "none"},
)
print(response.output_text)
print("input_tokens:", response.usage.input_tokens)
print("output_tokens:", response.usage.output_tokens)

미르시립도서관의 정기 휴관일은 **매주 월요일**입니다.  
출처: [E01 p.2]
input_tokens: 66
output_tokens: 37


## 5. 일부러 실패해 보기

전체 예산을 너무 작게 정하면 API 호출 전에 예외가 발생합니다. 오류를 확인한 뒤 정상 예산으로 돌아갑니다.

In [6]:
try:
    input_budget(300, 300, 128)
except ValueError as error:
    print("예상한 실패:", error)
else:
    raise AssertionError("예산 오류가 발생해야 합니다.")

예상한 실패: 출력 예약분과 여유가 전체 예산보다 큽니다.


## 정리

출력 공간을 먼저 예약하고, 입력을 로컬에서 추정한 뒤, 실제 사용량은 API 응답에서 확인합니다.